# Туториал 2: Оценка LLM на MMLU

Добро пожаловать во второй туториал нашего курса по оценке безопасности ИИ.

Оценка на бенчмарках — ключевой навык в прикладном машинном обучении, но статистическая
сторона часто остаётся на втором плане: публикуется одно число accuracy, а различия между
моделями считаются реальными без проверки того, могли ли они возникнуть случайно.
В этом туториале вы получите практический опыт запуска оценок с помощью библиотеки
inspect_ai и применения базовых статистических методов для строгой интерпретации результатов.

**Что вы изучите:**

- Загрузка и подготовка бенчмарк-датасета
- Вычисление доверительных интервалов для accuracy
- Статистическое сравнение моделей
- Анализ мощности для планирования размера оценки

**К концу туториала:** **У вас будет статистически строгий пайплайн оценки, который сможет сказать не только насколько точна модель, но и являются ли наблюдаемые различия между моделями реальными.**

## 1. Настройка

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from string import ascii_uppercase
from typing import Tuple, List

from inspect_ai import Task, task, eval
from inspect_ai.dataset import Sample, hf_dataset, FieldSpec
from inspect_ai.solver import multiple_choice
from inspect_ai.scorer import choice
from inspect_ai.log import EvalLog

In [3]:
# Настройка моделей -- замените на те, что доступны в вашем окружении.
# Примеры: 'ollama/llama3.2', 'openai/gpt-4o-mini', 'anthropic/claude-haiku-4-5'

MODEL_A = "ollama/llama2"        # слабая / базовая модель
MODEL_B = "ollama/qwen2:latest"  # сильная / модель для сравнения

## 2. Загрузка MMLU

`hf_dataset` — это загрузчик inspect_ai для датасетов с Hugging Face. Он скачивает данные
и оборачивает каждую запись в `Sample` — стандартный контейнер, который проходит через
весь пайплайн inspect_ai. `Sample` содержит входные данные для модели, ожидаемый ответ,
необязательные варианты ответов и произвольные метаданные, которые вы хотите сохранить.

MMLU хранит правильный ответ как целое число (0 = A, 1 = B, 2 = C, 3 = D).
Самый быстрый способ загрузить датасет — через `FieldSpec`, который сопоставляет имена
столбцов с полями `Sample`. Давайте попробуем и посмотрим, что получится.

In [13]:
dataset_raw = hf_dataset(
    path="cais/mmlu",
    name="all",
    split="test",
    sample_fields=FieldSpec(
        input="question",
        target="answer",           # raw MMLU answer is an integer index 0-3
        metadata=["choices", "subject"]
    ),
    cached=True
)

sample = dataset_raw[0]
print("input   :", sample.input[:80], "...")
print("target  :", sample.target,  "  <- integer index, not a letter!")
print("choices :", sample.metadata.get("choices"))

Значение `target` оказалось целым числом — но солвер `multiple_choice()` и скорер
`choice()` в inspect_ai ожидают букву (`"A"`, `"B"`, `"C"` или `"D"`).
Когда автоматического маппинга недостаточно, inspect_ai позволяет передать
**функцию record-to-sample**, которая получает полную исходную запись и возвращает
`Sample`, сконструированный вами.

In [14]:
def record_to_sample(record: dict) -> Sample:
    """
    Convert a raw MMLU record to an inspect_ai Sample.

    MMLU stores the correct answer as an integer index (0=A, 1=B, 2=C, 3=D).
    We convert it to the corresponding uppercase letter so it matches the
    format expected by the choice() scorer.
    """
    answer_idx = int(record["answer"])
    return Sample(
        input=record["question"],
        choices=record["choices"],
        target=ascii_uppercase[answer_idx],   # 0->'A', 1->'B', ...
        metadata=dict(subject=record.get("subject"))
    )


dataset = hf_dataset(
    path="cais/mmlu",
    name="all",
    split="test",
    sample_fields=record_to_sample,
    cached=True
)

sample = dataset[0]
print("target  :", sample.target, " <- letter now")
print("choices :", sample.choices)

## Задание 1: Создайте рабочее подмножество

Все эксперименты в этом ноутбуке будут выполняться на подмножестве по предметам,
достаточно маленьком для быстрой оценки. `Dataset.filter()` принимает предикат над
объектами `Sample`; поле `metadata` даёт доступ ко всему, что было задано в
`record_to_sample` — здесь это тег предмета MMLU.

Мы определяем `astronomy_subset` как референсный пример. Выберите любой предмет или
предметы из [списка предметов MMLU](https://huggingface.co/datasets/cais/mmlu#task-descriptions)
с не менее чем 50 вопросами, чтобы дальнейший анализ был статистически значимым. Создайте MY_SUBSET и используйте его во всех последующих упражнениях.

In [15]:
# Reference subset used in worked examples
astronomy_subset = dataset.filter(
    lambda s: s.metadata.get("subject") == "astronomy"
)
print(f"Astronomy: {len(astronomy_subset)} questions")


MY_SUBSET = # YOUR CODE HERE

print(f"My subset: {len(MY_SUBSET)} questions")

## 3. Запуск оценки

Каждая оценка в inspect_ai описывается объектом `Task`, который объединяет три вещи:

- **dataset** — вопросы
- **solver** — цепочка шагов, которая формирует ответ модели;
  `multiple_choice()` форматирует промпт с пронумерованными буквами вариантами и парсит выбор модели
- **scorer** — функция, которая оценивает ответ;
  `choice()` проверяет, совпадает ли выбранная буква с целевым ответом

Декоратор `@task` регистрирует функцию, чтобы inspect_ai мог найти её по имени
из CLI или передать напрямую в `eval()`.

In [7]:
@task
def mmlu_subset(subset):
    """Minimal MMLU task for any subject subset."""
    return Task(
        dataset=subset,
        solver=[multiple_choice()],
        scorer=choice()
    )

Вызов `eval()` запускает задачу и возвращает **список объектов `EvalLog`** — по одному
на пару (задача, модель). Всё, что вам нужно, находится в этом объекте; нет необходимости
читать лог-файлы с диска.

Два наиболее полезных атрибута:
- `log.results.scores` — список результатов скорера, каждый с словарём `metrics`
  (`"accuracy"`, `"stderr"` и т.д.)
- `log.samples` — список объектов `EvalSample` с входными данными, выходами и оценками для каждого вопроса

In [16]:
logs: List[EvalLog] = eval(
    mmlu_subset(astronomy_subset),
    model=MODEL_A,
    limit=10        # evaluate only the first 10 questions
)

log = logs[0]      # one task -> one log
print("Status  :", log.status)
print("Model   :", log.eval.model)
print("Accuracy:", log.results.scores[0].metrics["accuracy"].value)

## 4. Из `EvalLog` в DataFrame

## Задание 2: Реализуйте `log_to_df`

Агрегированная accuracy в `log.results` полезна для быстрой проверки, но для
статистического анализа далее нам нужна плоская таблица: **одна строка на (вопрос, эпоху)**
с числовым столбцом `score`.

`log.samples` — это список объектов `EvalSample`. Каждый из них содержит:
- `.id` — идентификатор вопроса
- `.epoch` — к какому прогону он относится (актуально при `epochs > 1`)
- `.scores` — словарь, сопоставляющий имя скорера с `Score`; значение `Score.value` для `choice()` —
  `"C"` (correct, правильно) или `"I"` (incorrect, неправильно)
- `.metadata` — словарь метаданных, заданный в `record_to_sample`

Реализуйте `log_to_df` так, чтобы она преобразовывала `EvalLog` в DataFrame со столбцами
`id`, `epoch`, `score` (1/0) и `subject`. Тест ниже проверит структуру.

In [17]:
def log_to_df(log: EvalLog) -> pd.DataFrame:
    """
    Convert an EvalLog to a DataFrame with one row per (question, epoch).

    Columns:
        id      – question identifier
        epoch   – epoch index (0 if epochs=1)
        score   – 1 if correct, 0 otherwise
        subject – MMLU subject tag from metadata

    The choice() scorer stores the result as "C" (correct) or "I" (incorrect).
    """
    # YOUR CODE HERE
    raise NotImplementedError

# =================================== TESTS ===================================
df_test = log_to_df(log)

assert set(df_test.columns) >= {"id", "epoch", "score", "subject"}
assert df_test["score"].isin([0, 1]).all()
assert len(df_test) == 10

print(df_test.head())
print(f"\nAccuracy: {df_test['score'].mean():.1%}")

## 5. Доверительные интервалы


Одно число accuracy несёт в себе неопределённость: оценка использовала конечный набор
вопросов, выбранных из гораздо большего пространства. Статья (§2.1, §3.1) показывает,
как количественно оценить это с помощью стандартной ошибки по ЦПТ.


## Задание 3: Реализуйте `ci_accuracy_basic` и `ci_accuracy`

**`ci_accuracy_basic(scores, ci)`** — простой случай, когда каждый вопрос отвечен
ровно один раз. `scores` — обычный numpy-массив из 0 и 1. Используйте формулу 1 из статьи.

**`ci_accuracy(df, ci)`** — общий случай, обрабатывающий несколько прогонов на вопрос
(`epochs > 1`). Когда для вопроса существует K прогонов, сначала усредните их оценки,
затем вычислите SE по средним на уровне вопросов. Объединение всех K×n индивидуальных
ответов занизило бы дисперсию — ответы на один и тот же вопрос в разных эпохах коррелированы.

In [ ]:
def ci_accuracy_basic(scores: np.ndarray, ci: float = 0.95) -> Tuple[float, float, float]:
    """
    CLT-based confidence interval for accuracy -- single run per question (K = 1).

    Parameters
    ----------
    scores : 1-D array of per-question binary scores (0 or 1)
    ci     : confidence level (default 0.95)

    Returns
    -------
    (lower_bound, mean_accuracy, upper_bound)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def ci_accuracy(df: pd.DataFrame, ci: float = 0.95) -> Tuple[float, float, float]:
    """
    CLT-based confidence interval for accuracy, supporting multiple epochs (K >= 1).

    Parameters
    ----------
    df : DataFrame returned by log_to_df, with columns 'id', 'score', 'epoch'
    ci : confidence level (default 0.95)

    Returns
    -------
    (lower_bound, mean_accuracy, upper_bound)
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# =================================== TESTS ===================================
def _make_df(ids, scores, epochs=None):
    if epochs is None:
        epochs = [0] * len(ids)
    return pd.DataFrame({"id": ids, "score": scores, "epoch": epochs})

# ci_accuracy_basic
l, m, u = ci_accuracy_basic(np.ones(10))

assert l == 1.0 and u == 1.0, "perfect accuracy: CI should collapse to 1"

l, m, u = ci_accuracy_basic(np.zeros(10))

assert l == 0.0 and u == 0.0, "zero accuracy: CI should collapse to 0"

scores3 = np.array([1, 1, 0, 1, 0], dtype=float)
l, m, u = ci_accuracy_basic(scores3)

assert l < 0.6 < u, f"0.6 not in [{l:.3f}, {u:.3f}]"

np.random.seed(42)
s = np.random.binomial(1, 0.75, 200).astype(float)
l95, _, u95 = ci_accuracy_basic(s, 0.95)
l99, _, u99 = ci_accuracy_basic(s, 0.99)

assert (u99 - l99) > (u95 - l95), "99% CI must be wider than 95%"
assert np.isclose(l95, 0.6819421067148456)
assert np.isclose(u95, 0.8080578932851544)

# ci_accuracy (K=1 should match basic)
df3 = _make_df([1,2,3,4,5], scores3.tolist())
l_df, _, u_df = ci_accuracy(df3)
l_ar, _, u_ar = ci_accuracy_basic(scores3)

assert np.isclose(l_df, l_ar) and np.isclose(u_df, u_ar), "K=1 must match basic version"

# ci_accuracy (K=3 should give narrower CI on average)
np.random.seed(0)
rows_k1, rows_k3 = [], []
for q in range(30):
    p = np.random.uniform(0.3, 0.9)
    rows_k1.append({"id": q, "score": int(np.random.binomial(1, p)), "epoch": 0})
    for e in range(3):
        rows_k3.append({"id": q, "score": int(np.random.binomial(1, p)), "epoch": e})

l1, _, u1 = ci_accuracy(pd.DataFrame(rows_k1))
l3, _, u3 = ci_accuracy(pd.DataFrame(rows_k3))
print(f"K=1 width: {u1-l1:.3f}")
print(f"K=3 width: {u3-l3:.3f}  (narrower on average)")
print("\n✓ All tests passed!")

## 6. Визуализация сужения доверительных интервалов

Два фактора делают доверительные интервалы уже: больше вопросов (больше n) и больше
прогонов на вопрос (больше K). Ваша задача — визуализировать эти эффекты.

## Задание 4.1: Постройте график ширины ДИ в зависимости от числа эпох

In [25]:
k_values    = [5, 6, 7, 8, 9, 10]
accuracies    = []
ci_lowers     = []
ci_uppers     = []

# YOUR CODE HERE

plt.figure(figsize=(8, 4))
plt.fill_between(k_values, ci_lowers, ci_uppers, alpha=0.25, label="95% CI")
plt.plot(k_values, accuracies, "o-", lw=2, label="Accuracy")
plt.xlabel("Number of runs per question (K)")
plt.ylabel("Accuracy")
plt.title(f"{MODEL_A} on MMLU-subset — accuracy and CI vs k")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

---
1. Посмотрите, как быстро полоса сужается.
   В какой момент запуск ещё одной эпохи перестаёт окупаться?
2. Увеличение K меняет вашу оценку accuracy модели или только вашу уверенность в ней?
3. Что это говорит о том, как распределять бюджет на оценку?

**Ваш ответ:**

## Задание 4.2: Вычислите и постройте график ширины ДИ в зависимости от n

Для каждого размера выборки n из `range(10, len(question_ids)+1, 10)` возьмите срез
обоих DataFrame по первым n идентификаторам вопросов, вычислите `ci_accuracy` и
запишите ширину ДИ. Затем постройте график ширины от n.

In [ ]:
question_ids  = # YOUR CODE HERE
dataset_sizes = range(10, len(question_ids) + 1, 10)
accuracies    = []
ci_lowers     = []
ci_uppers     = []

# YOUR CODE HERE

plt.figure(figsize=(8, 4))
plt.fill_between(dataset_sizes, ci_lowers, ci_uppers, alpha=0.25, label="95% CI")
plt.plot(dataset_sizes, accuracies, "o-", lw=2, label="Accuracy")
plt.xlabel("Number of questions (n)")
plt.ylabel("Accuracy")
plt.title(f"{MODEL_A} on MMLU-subset — accuracy and CI vs n")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

---
1. При каком n линия accuracy начинает выглядеть стабильной?
2. Сравните это число с размером `MY_SUBSET` — находитесь ли вы в надёжной области?
3. Сравните эту кривую с кривой из 4.1. В чём разница между тем, что дают K и n?

**Ваш ответ:**

## 7. Сравнение двух моделей

Простое сопоставление двух чисел accuracy не говорит о том, является ли разрыв
реальным или просто шумом. Статья (§4.2) описывает **парный тест**: поскольку обе
модели отвечают на одни и те же вопросы, можно вычислить поквопросные разности
оценок и проверить, отличается ли их среднее значимо от нуля. Это устраняет
дисперсию сложности вопросов и даёт меньшую стандартную ошибку, чем если бы два
прогона рассматривались как независимые выборки.

## Задание 5: Сравните две модели

`run_and_get_scores` и `compare_models_paired` предоставлены. Допишите
`significance_by_paired_ttest` и используйте её для сравнения двух моделей на `MY_SUBSET`.

Реализуйте `significance_by_paired_ttest` и сравните MODEL_A и MODEL_B.

In [15]:
def run_and_get_scores(model_name: str, dataset, epochs: int = 1) -> np.ndarray:
    """Run eval and return mean-per-question scores, sorted by question id."""
    print(f"  Running {model_name} ...")
    run_logs = eval(mmlu_subset(dataset), model=model_name, epochs=epochs)
    df = log_to_df(run_logs[0])
    return df.groupby("id")["score"].mean().sort_index().values


def significance_by_paired_ttest(
    scores1: np.ndarray,
    scores2: np.ndarray,
    alpha: float = 0.05,
    two_tailed: bool = True,
) -> Tuple[float, float, bool]:
    """
    Paired t-test between two sets of per-question scores.

    Returns (p_value, mean_difference scores1 - scores2, is_significant).
    """
    assert len(scores1) == len(scores2), "arrays must cover the same questions"
    
    alternative = "two-sided" if two_tailed else "greater"
    
    _, p_value  = # YOUR CODE HERE
    mean_diff   = # YOUR CODE HERE
    
    return p_value, mean_diff, bool(p_value < alpha)


def compare_models_paired(
    model_a: str,
    model_b: str,
    dataset,
    alpha: float = 0.05,
    two_tailed: bool = True,
    epochs_a: int = 1,
    epochs_b: int = 1,
) -> Tuple[float, float, bool]:
    """
    Evaluate both models on the same dataset and run a paired t-test.

    Returns (p_value, mean_difference A - B, is_significant).
    """
    scores_a = run_and_get_scores(model_a, dataset, epochs=epochs_a)
    scores_b = run_and_get_scores(model_b, dataset, epochs=epochs_b)
    return significance_by_paired_ttest(scores_a, scores_b, alpha, two_tailed)

In [18]:
# =================================== TESTS ===================================
p, d, sig = significance_by_paired_ttest(np.array([1,2,3]), np.array([1,2,3]))

assert np.isclose(d, 0.0) and not sig

p, d, sig = significance_by_paired_ttest(
    np.array([1,1,1,1,1]), np.array([0,0,0,0,0]), two_tailed=False
)

assert sig and d > 0

print("All tests passed!")

In [17]:
# YOUR CODE HERE

---
1. Запишите полученные p-value и среднюю разность.
2. Является ли разрыв значимым? Достаточно ли он велик, чтобы иметь практическое значение?
3. Что изменило бы ваш вывод: больше вопросов, другой предмет или другая пара моделей?

**Ваш ответ:**

## 8. Интервальная оценка разности accuracy

В задании 5 вы получили решение «да/нет» о значимости. Здесь вы оцените величину
разрыва и его неопределённость: доверительный интервал для разности даёт обе части
информации одновременно.

## Задание 6: Оцените разрыв в accuracy

Используйте `ci_accuracy_basic` для вычисления 95%-го ДИ по поквопросным разностям оценок.

Вычислите и сообщите доверительный интервал для MODEL_A − MODEL_B.

In [18]:
# YOUR CODE HERE

---
1. Запишите интервал. Содержит ли он ноль?
2. Как это соотносится с тестом значимости из задания 5 — рассказывают ли они одну и ту же историю?
3. Какой результат более информативен — p-value или интервал? Почему?

**Ваш ответ:**

## 9. Анализ мощности

Прежде чем запускать дорогостоящую оценку, стоит спросить: сколько вопросов нам нужно,
чтобы обнаружить значимое различие с достаточной статистической мощностью?
Статья (§5) выводит минимальный обнаруживаемый эффект как функцию размера выборки n,
дисперсии на уровне вопросов ω² и внутримодельной дисперсии σ².

## Задание 7: Оцените компоненты дисперсии

Реализуйте `estimate_variance_components` и сообщите MDE для `MY_SUBSET` при α = 0.05, мощность = 80%.

In [19]:
def estimate_variance_components(
    logs_a: List[EvalLog],
    logs_b: List[EvalLog],
) -> dict:
    """
    Estimate omega2, sigma2_a, sigma2_b from two EvalLog objects (see ss5 of the paper).

    Both logs must cover the same set of questions. Use epochs >= 2 so that
    within-question variance can be estimated.

    Returns dict with keys 'omega2', 'sigma2_a', 'sigma2_b'.
    """

    # YOUR CODE HERE

    return {
        "omega2":   ...,
        "sigma2_a": ...,
        "sigma2_b": ...,
    }


def minimum_detectable_effect(
    n: int,
    omega2: float,
    sigma2_a: float = 0.0,
    sigma2_b: float = 0.0,
    ka: int = 1,
    kb: int = 1,
    alpha: float = 0.05,
    power: float = 0.80,
) -> float:
    """MDE for a paired model comparison (Eq. 10 in the paper)."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta  = stats.norm.ppf(power)
    return float((z_alpha + z_beta) * np.sqrt(
        (omega2 + sigma2_a / ka + sigma2_b / kb) / n
    ))

In [19]:
print("Running pilot evals ...")
logs_a = eval(mmlu_subset(MY_SUBSET), model=MODEL_A, epochs=2, limit=15)
logs_b = eval(mmlu_subset(MY_SUBSET), model=MODEL_B, epochs=2, limit=15)

params = estimate_variance_components(logs_a, logs_b)
print(f"omega2  = {params['omega2']:.4f}")
print(f"sigma2_A = {params['sigma2_a']:.4f}")
print(f"sigma2_B = {params['sigma2_b']:.4f}")

mde = minimum_detectable_effect(n=len(MY_SUBSET), **params)
print(f"\nWith n={len(MY_SUBSET)} questions -> MDE = {mde:.1%}")
print("(smallest gap detectable at 80% power, alpha=0.05)")

---
1. Какой MDE вы получили для `MY_SUBSET`? Является ли этот разрыв практически значимым?
2. Если MDE больше, чем разрыв, наблюдённый в задании 5,
   что это говорит о вашем предыдущем результате?

**Ваш ответ:**

## Задание 8: Реализуйте `required_sample_size`

`minimum_detectable_effect` вычисляет delta по n. Реализуйте обратную функцию:
по заданному целевому delta верните минимально необходимое n. Используйте формулу
размера выборки из §5 статьи (формула 9). Убедитесь, что функция проходит проверку
«туда-обратно» (round-trip check), затем используйте её для вычисления количества
вопросов, необходимых для обнаружения разрыва в 5% и 10% accuracy на `MY_SUBSET`.

In [ ]:
# --- Assignment 7 -----------------------------------------------------------
def required_sample_size(
    delta: float,
    omega2: float,
    sigma2_a: float = 0.0,
    sigma2_b: float = 0.0,
    ka: int = 1,
    kb: int = 1,
    alpha: float = 0.05,
    power: float = 0.80,
) -> int:
    """Minimum number of questions needed to detect `delta` at the given power."""
    # YOUR CODE HERE
    raise NotImplementedError


# =================================== TESTS ===================================
n_needed = required_sample_size(delta=0.05, **params)
print(f"Questions needed to detect delta=5%: {n_needed}")

mde_check = minimum_detectable_effect(n=n_needed, **params)

assert abs(mde_check - 0.05) < 0.005, f"Round-trip failed: MDE={mde_check:.3f}"

print("Round-trip check passed!")

In [1]:
# YOUR CODE HERE

---
1. Сколько вопросов нужно для обнаружения разрыва в 5%? А в 10%?
2. Достаточно ли в `MY_SUBSET` вопросов, чтобы быть полезным бенчмарком для сравнения этих двух моделей?

**Ваш ответ:**

## Задание 9: Сравните модель с самой собой: baseline vs chain-of-thought

Солвер `multiple_choice()`, который мы использовали до сих пор, предлагает модели
отвечать напрямую. inspect_ai также предоставляет `chain_of_thought`, который просит
модель рассуждать пошагово перед тем, как дать окончательный ответ.

Используя инфраструктуру парного сравнения из раздела 7, оцените одну и ту же модель
дважды на одном подмножестве — один раз с солвером по умолчанию и один раз с
`chain_of_thought` — и проверьте, является ли разница в accuracy статистически
значимой. Помогает ли рассуждение? Является ли эффект устойчивым для разных предметов?

In [ ]:
# YOUR CODE HERE

---
1. Помогает ли chain-of-thought? Насколько и является ли эффект значимым?
2. Удивляет ли вас результат? Что может его объяснить?
3. Ожидали бы вы такой же паттерн на другом предмете?

**Ваш ответ:**

## Бонусное задание: Кластерные стандартные ошибки

Некоторые бенчмарки содержат группы связанных вопросов — например, несколько вопросов
по одному и тому же тексту в задачах на понимание прочитанного, таких как DROP или RACE.
В таких случаях стандартный доверительный интервал по ЦПТ является антиконсервативным:
вопросы внутри группы коррелированы, поэтому эффективный размер выборки меньше n.
Miller (2024) решает эту проблему с помощью кластерных стандартных ошибок (§2.2).

Используя бенчмарк на понимание прочитанного по вашему выбору, реализуйте кластерный
доверительный интервал (формула 4 из статьи) и сравните его с наивным интервалом по ЦПТ.
Насколько шире кластерный интервал? Зависит ли разница от бенчмарка?
Затем сравните две модели на том же бенчмарке, используя кластерную парную стандартную
ошибку (формула 8) — изменится ли вывод о том, какая модель лучше, по сравнению
с наивным парным тестом?

In [ ]:
# YOUR CODE HERE